In [ ]:
"""
================================================================================
COMPLETE RESEARCH CODE - CONSERVATIVE NOISE AUGMENTATION FOR UAV LOCALIZATION
================================================================================

This is a single, complete file with everything:
- Dual-camera data loading (cam0 + cam1)
- Conservative noise augmentation (5 types)
- Stable model architecture
- Training pipeline
- Full evaluation on 5 noise conditions
- Results tables and plots

Just run: python complete_research.py

Author: Shamsiddinov Nodir
Date: 2025
================================================================================
"""

import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from scipy.spatial.transform import Rotation
import matplotlib.pyplot as plt
import time
from typing import Dict, Tuple, List
import random
import json


# ============================================================================
# PART 1: DATASET LOADING (DUAL CAMERA)
# ============================================================================

class DualCameraEuRoCDataset(Dataset):
    """
    Load EuRoC dataset from BOTH cam0 and cam1.
    Doubles the dataset size from ~3,682 to ~7,364 images.
    """
    
    def __init__(self, root_dir, camera='cam0', transform=None, max_samples=None):
        self.root_dir = root_dir
        self.camera = camera
        self.transform = transform
        
        # Paths
        self.image_dir = os.path.join(root_dir, camera, 'data')
        self.gt_file = os.path.join(root_dir, 'state_groundtruth_estimate0', 'data.csv')
        
        print(f"  Loading {camera} from {root_dir}...")
        
        # Load ground truth
        self.gt_data = self._load_groundtruth()
        
        # Load images
        self.image_files = sorted([f for f in os.listdir(self.image_dir) 
                                   if f.endswith('.png')])
        
        # Match to poses
        self.data = self._match_images_to_poses()
        
        if max_samples:
            self.data = self.data[:max_samples]
        
        print(f"    ✓ Loaded {len(self.data)} samples from {camera}")
    
    def _load_groundtruth(self):
        """Load ground truth CSV"""
        gt_df = pd.read_csv(self.gt_file, header=0)
        return {
            'timestamp': gt_df.iloc[:, 0].values,
            'position': gt_df.iloc[:, 1:4].values,
            'quaternion': gt_df.iloc[:, 4:8].values
        }
    
    def _match_images_to_poses(self):
        """Match images to closest ground truth pose"""
        matched_data = []
        gt_timestamps = self.gt_data['timestamp']
        
        for img_file in self.image_files:
            img_timestamp = int(img_file.split('.')[0])
            idx = np.argmin(np.abs(gt_timestamps - img_timestamp))
            
            position = self.gt_data['position'][idx]
            quaternion = self.gt_data['quaternion'][idx]
            
            # Convert quaternion to Euler angles
            rot = Rotation.from_quat([quaternion[1], quaternion[2], 
                                      quaternion[3], quaternion[0]])
            euler = rot.as_euler('xyz', degrees=False)
            
            pose = np.concatenate([position, euler]).astype(np.float32)
            
            matched_data.append({
                'image_file': img_file,
                'pose': pose,
                'camera': self.camera
            })
        
        return matched_data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, sample['image_file'])
        image = Image.open(img_path).convert('RGB')
        
        # Get pose
        pose = torch.from_numpy(sample['pose'])
        
        # Transform
        if self.transform:
            image = self.transform(image)
        
        return image, pose


def create_dual_camera_dataloaders(dataset_path, batch_size=32, num_workers=0):
    """
    Create dataloaders using BOTH cameras with augmentation.
    Returns ~10,000 effective training samples.
    """
    
    print("\n" + "="*70)
    print("STEP 1: LOADING DUAL-CAMERA DATASET")
    print("="*70 + "\n")
    
    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=5),
        transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Load BOTH cameras
    print("Loading cameras:")
    cam0_dataset = DualCameraEuRoCDataset(dataset_path, camera='cam0', transform=None)
    cam1_dataset = DualCameraEuRoCDataset(dataset_path, camera='cam1', transform=None)
    
    # Combine
    combined_data = cam0_dataset.data + cam1_dataset.data
    print(f"\n  ✓ Total samples: {len(combined_data)}")
    
    # Shuffle and split
    random.shuffle(combined_data)
    
    total_size = len(combined_data)
    train_size = int(0.70 * total_size)
    val_size = int(0.15 * total_size)
    
    train_data = combined_data[:train_size]
    val_data = combined_data[train_size:train_size + val_size]
    test_data = combined_data[train_size + val_size:]
    
    print(f"  ✓ Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
    print(f"  ✓ Effective training size with augmentation: ~{len(train_data) * 2}")
    
    # Create datasets
    class SimpleDataset(Dataset):
        def __init__(self, data, root_dir, transform):
            self.data = data
            self.root_dir = root_dir
            self.transform = transform
        
        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            sample = self.data[idx]
            img_path = os.path.join(self.root_dir, sample['camera'], 'data', 
                                   sample['image_file'])
            image = Image.open(img_path).convert('RGB')
            pose = torch.from_numpy(sample['pose'])
            
            if self.transform:
                image = self.transform(image)
            
            return image, pose
    
    train_dataset = SimpleDataset(train_data, dataset_path, train_transform)
    val_dataset = SimpleDataset(val_data, dataset_path, train_transform)
    test_dataset = SimpleDataset(test_data, dataset_path, test_transform)
    
    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                           shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=num_workers, pin_memory=True)
    
    print("="*70 + "\n")
    
    return train_loader, val_loader, test_loader


# ============================================================================
# PART 2: CONSERVATIVE NOISE AUGMENTATION
# ============================================================================

class ConservativeNoiseAugmentation:
    """
    Conservative noise models with 40% reduced intensities.
    """
    
    def __init__(self, seed=None):
        if seed:
            np.random.seed(seed)
            torch.manual_seed(seed)
    
    def gaussian_noise(self, image: torch.Tensor, sigma: float = 10.0) -> torch.Tensor:
        """Gaussian noise: σ=15 (reduced from 25)"""
        noise = torch.randn_like(image) * (sigma / 255.0)
        return torch.clamp(image + noise, -2, 2)
    
    def poisson_noise(self, image: torch.Tensor, scale: float = 200.0) -> torch.Tensor:
        """Poisson noise: scale=75 (reduced from 50)"""
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(image.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(image.device)
        
        image_denorm = image * std + mean
        image_denorm = torch.clamp(image_denorm, 0, 1)
        image_scaled = image_denorm * 255.0
        
        if image.is_cuda:
            image_np = image_scaled.cpu().numpy()
            noisy_np = np.random.poisson(image_np / scale) * scale
            noisy_scaled = torch.from_numpy(noisy_np).float().to(image.device)
        else:
            image_np = image_scaled.numpy()
            noisy_np = np.random.poisson(image_np / scale) * scale
            noisy_scaled = torch.from_numpy(noisy_np).float()
        
        noisy_image = torch.clamp(noisy_scaled / 255.0, 0, 1)
        noisy_image = (noisy_image - mean) / std
        
        return noisy_image
    
    def motion_blur(self, image: torch.Tensor, kernel_size: int = 5) -> torch.Tensor:
        """Motion blur: kernel=9 (reduced from 15)"""
        kernel = np.zeros((kernel_size, kernel_size))
        kernel[int((kernel_size - 1) / 2), :] = np.ones(kernel_size)
        kernel = kernel / kernel_size
        
        kernel = torch.from_numpy(kernel).float().unsqueeze(0).unsqueeze(0).to(image.device)
        kernel = kernel.expand(3, 1, kernel_size, kernel_size)
        
        image_unsqueezed = image.unsqueeze(0)
        blurred = torch.nn.functional.conv2d(image_unsqueezed, kernel, 
                                            padding=kernel_size // 2, groups=3)
        return blurred.squeeze(0)
    
    def spatially_correlated_noise(self, image: torch.Tensor, 
                                   sigma: float = 10.0, 
                                   correlation_length: int = 2) -> torch.Tensor:
        """Spatially correlated noise: σ=15, corr=2 (reduced from 25, 3)"""
        noise = torch.randn_like(image) * (sigma / 255.0)
        
        kernel_size = correlation_length * 2 + 1
        x = np.arange(kernel_size) - correlation_length
        kernel_1d = np.exp(-x**2 / (2 * correlation_length**2))
        kernel_2d = np.outer(kernel_1d, kernel_1d)
        kernel_2d = kernel_2d / kernel_2d.sum()
        
        kernel = torch.from_numpy(kernel_2d).float().unsqueeze(0).unsqueeze(0).to(image.device)
        kernel = kernel.expand(3, 1, kernel_size, kernel_size)
        
        noise_unsqueezed = noise.unsqueeze(0)
        correlated_noise = torch.nn.functional.conv2d(noise_unsqueezed, kernel,
                                                      padding=kernel_size // 2, groups=3)
        correlated_noise = correlated_noise.squeeze(0)
        
        return torch.clamp(image + correlated_noise, -3, 3)
    
    def no_noise(self, image: torch.Tensor) -> torch.Tensor:
        """Identity - no noise"""
        return image


# ============================================================================
# PART 3: STABLE MODEL ARCHITECTURE
# ============================================================================

class StablePoseEstimator(nn.Module):
    """
    Stable architecture: ResNet18 (frozen) + simple head with BatchNorm.
    Only ~4,000 trainable parameters.
    """
    
    def __init__(self, pretrained=True):
        super(StablePoseEstimator, self).__init__()
        
        self.backbone = resnet18(weights=pretrained)
        
        # FREEZE backbone
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Simple regression head
        self.backbone.fc = nn.Sequential(
            nn.Linear(512, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.7),
            nn.Linear(64, 6)
        )
        
        # Initialize with small weights
        nn.init.xavier_uniform_(self.backbone.fc[0].weight, gain=0.01)
        nn.init.xavier_uniform_(self.backbone.fc[4].weight, gain=0.01)
        nn.init.zeros_(self.backbone.fc[0].bias)
        nn.init.zeros_(self.backbone.fc[4].bias)
    
    def forward(self, x):
        return self.backbone(x)
    
    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================================
# PART 4: LOSS FUNCTION
# ============================================================================

class ConservativePoseLoss(nn.Module):
    """Conservative loss: β=10 (translation-focused)"""
    
    def __init__(self, beta=10.0):
        super(ConservativePoseLoss, self).__init__()
        self.beta = beta
    
    def forward(self, predictions, targets):
        pred_trans = predictions[:, :3]
        pred_rot = predictions[:, 3:]
        
        target_trans = targets[:, :3]
        target_rot = targets[:, 3:]
        
        trans_loss = torch.norm(pred_trans - target_trans, dim=1).mean()
        rot_loss = torch.norm(pred_rot - target_rot, dim=1).mean()
        
        total_loss = trans_loss + self.beta * rot_loss
        
        return total_loss, {
            'total': total_loss.item(),
            'translation': trans_loss.item(),
            'rotation': rot_loss.item()
        }


# ============================================================================
# PART 5: TRAINING FUNCTION
# ============================================================================

def train_model(model, train_loader, val_loader, lr=5e-4, patience=10,
                noise_fn=None, model_name='model',
                num_epochs=30, device='cuda'):
    """
    Train one model with ultra-conservative settings.
    """
    
    print(f"\n{'='*70}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*70}")
    
    model = model.to(device)
    
    criterion = ConservativePoseLoss(beta=10.0)
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-3
    )
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.3, patience=2, verbose=True, min_lr=1e-7
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    patience = patience
    
    history = {'train_loss': [], 'val_loss': []}
    
    print(f"  Trainable params: {model.get_num_parameters():,}")
    print(f"  Learning rate: {lr} | Weight decay: 1e-3 | Beta: 10 | Patience: {patience}")
    print()
    
    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        train_loss = 0.0
        for images, poses in train_loader:
            images, poses = images.to(device), poses.to(device)
            
            if noise_fn:
                images = torch.stack([noise_fn(img) for img in images])
            
            optimizer.zero_grad()
            predictions = model(images)
            loss, _ = criterion(predictions, poses)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, poses in val_loader:
                if noise_fn:
                    images = torch.stack([noise_fn(img) for img in images])
                images, poses = images.to(device), poses.to(device)
                predictions = model(images)
                loss, _ = criterion(predictions, poses)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        # Check improvement
        if val_loss < best_val_loss - 0.01:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f'checkpoints/{model_name}_best.pth')
            improvement = "✓ IMPROVED"
        else:
            patience_counter += 1
            improvement = f"✗ No improvement ({patience_counter}/{patience})"
        
        print(f"Epoch [{epoch:2d}/{num_epochs}] | "
              f"Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"{improvement}")
        
        scheduler.step(val_loss)
        
        if patience_counter >= patience:
            print(f"\n  Early stopping at epoch {epoch} (Best val: {best_val_loss:.4f})")
            break
    
    # Load best model
    model.load_state_dict(torch.load(f'checkpoints/{model_name}_best.pth'))
    
    print(f"✓ Training completed (Best val loss: {best_val_loss:.4f})")
    print("="*70 + "\n")
    
    return model, history


# ============================================================================
# PART 6: EVALUATION FUNCTION
# ============================================================================

def evaluate_model(model, test_loader, noise_fn, device, condition_name):
    """
    Evaluate model on test set with specific noise condition.
    """
    model.eval()
    criterion = ConservativePoseLoss(beta=10.0)
    
    total_trans_error = 0.0
    total_rot_error = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for images, poses in test_loader:
            images, poses = images.to(device), poses.to(device)
            
            # Apply test noise
            if noise_fn is not None:
                images = torch.stack([noise_fn(img) for img in images])
            
            predictions = model(images)
            _, loss_dict = criterion(predictions, poses)
            
            total_trans_error += loss_dict['translation']
            total_rot_error += loss_dict['rotation']
            num_batches += 1
    
    avg_trans = total_trans_error / num_batches
    avg_rot = total_rot_error / num_batches
    
    return {
        'translation_error': avg_trans,
        'rotation_error': avg_rot
    }

In [ ]:
def save_results(results, histories, save_dir):
    """
    Save all results: tables, plots, JSON.
    """

    # 1. Save results table
    with open(f'{save_dir}/results_table.txt', 'w') as f:
        f.write("="*85 + "\n")
        f.write("CONSERVATIVE NOISE AUGMENTATION - EVALUATION RESULTS\n")
        f.write("="*85 + "\n\n")
        
        f.write(f"{'Model':<20s} | {'Clean':<8s} | {'Low':<8s} | {'Med':<8s} | {'High':<8s} | {'Real':<8s}\n")
        f.write("-"*85 + "\n")
        
        for model_name, conditions in results.items():
            f.write(f"{model_name:<20s}")
            for cond in ['clean', 'low_noise', 'medium_noise', 'high_noise', 'real_sensor']:
                error = conditions[cond]['translation_error']
                f.write(f" | {error:>6.4f}m")
            f.write("\n")
        
        f.write("\nLower is better. Values are translation errors in meters.\n")

    print("  ✓ Saved results_table.txt")

    # 2. Save JSON
    json_results = {}
    for model_name, conditions in results.items():
        json_results[model_name] = {}
        for cond_name, metrics in conditions.items():
            json_results[model_name][cond_name] = {
                'translation_error': float(metrics['translation_error']),
                'rotation_error': float(metrics['rotation_error'])
            }

    with open(f'{save_dir}/results.json', 'w') as f:
        json.dump(json_results, f, indent=2)

    print("  ✓ Saved results.json")

    # 3. Training curves
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, (name, history) in enumerate(histories.items()):
        ax = axes[idx]
        epochs = range(1, len(history['train_loss']) + 1)
        ax.plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
        ax.plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.set_title(name.replace('_', ' ').title(), fontweight='bold', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        best_epoch = np.argmin(history['val_loss'])
        ax.axvline(x=best_epoch+1, color='green', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    print("  ✓ Saved training_curves.png")

    # 4. Robustness comparison
    fig, ax = plt.subplots(figsize=(12, 6))

    conditions = ['clean', 'low_noise', 'medium_noise', 'high_noise', 'real_sensor']
    condition_labels = ['Clean', 'Low', 'Medium', 'High', 'Real Sensor']

    x = np.arange(len(conditions))
    width = 0.13

    colors = ['gray', 'blue', 'orange', 'green', 'red', 'purple']

    for idx, (model_name, color) in enumerate(zip(results.keys(), colors)):
        errors = [results[model_name][cond]['translation_error'] for cond in conditions]
        offset = (idx - 2.5) * width
        ax.bar(x + offset, errors, width, label=model_name, color=color, alpha=0.8)

    ax.set_xlabel('Test Condition', fontsize=12)
    ax.set_ylabel('Translation Error (meters)', fontsize=12)
    ax.set_title('Model Robustness Across Noise Conditions', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(condition_labels)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{save_dir}/robustness_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    print("  ✓ Saved robustness_comparison.png")

    # 5. Summary statistics
    with open(f'{save_dir}/summary_statistics.txt', 'w') as f:
        f.write("="*70 + "\n")
        f.write("SUMMARY STATISTICS\n")
        f.write("="*70 + "\n\n")
        
        # Find best models
        noisy_conditions = ['low_noise', 'medium_noise', 'high_noise', 'real_sensor']
        
        avg_errors = {}
        for model_name, conditions in results.items():
            avg_error = np.mean([conditions[cond]['translation_error'] 
                                for cond in noisy_conditions])
            avg_errors[model_name] = avg_error
        
        best_model = min(avg_errors, key=avg_errors.get)
        
        f.write(f"Best Overall Model: {best_model}\n")
        f.write(f"  Average error on noisy data: {avg_errors[best_model]:.4f}m\n\n")
        
        # Best on each condition
        f.write("Best Model Per Condition:\n")
        for cond, label in zip(['clean', 'low_noise', 'medium_noise', 'high_noise', 'real_sensor'],
                                ['Clean', 'Low Noise', 'Medium Noise', 'High Noise', 'Real Sensor']):
            errors = {name: results[name][cond]['translation_error'] 
                        for name in results.keys()}
            best = min(errors, key=errors.get)
            f.write(f"  {label:15s}: {best:20s} ({errors[best]:.4f}m)\n")
        
        f.write("\n")
        
        # Improvement over baseline
        f.write("Improvement Over Baseline:\n")
        baseline_errors = {cond: results['baseline'][cond]['translation_error'] 
                            for cond in noisy_conditions}
        
        for model_name in results.keys():
            if model_name == 'baseline':
                continue
            
            improvements = []
            for cond in noisy_conditions:
                baseline_err = baseline_errors[cond]
                model_err = results[model_name][cond]['translation_error']
                improvement = ((baseline_err - model_err) / baseline_err) * 100
                improvements.append(improvement)
            
            avg_improvement = np.mean(improvements)
            f.write(f"  {model_name:20s}: {avg_improvement:+6.1f}% average\n")
        
        f.write("\n")
        
        # Training stability
        f.write("Training Stability (Best Epoch):\n")
        for model_name, history in histories.items():
            best_epoch = np.argmin(history['val_loss']) + 1
            total_epochs = len(history['val_loss'])
            f.write(f"  {model_name:20s}: Epoch {best_epoch:2d}/{total_epochs}\n")

    print("  ✓ Saved summary_statistics.txt")
    print()




In [ ]:
trained_models = {}
histories = {}
results = {}

In [ ]:
# ============================================================================
# PART 7: MAIN PIPELINE
# ============================================================================
"""
Complete research pipeline with full evaluation.
"""
print("\n" + "="*70)
print("CONSERVATIVE NOISE AUGMENTATION FOR UAV LOCALIZATION")
print("Complete Research Pipeline V2.0")
print("="*70)

# Configuration
DATASET_PATH = r"C:\Users\Asus\Downloads\MH_01_easy\mav0"
BATCH_SIZE = 32
NUM_EPOCHS = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nDevice: {DEVICE}")
if DEVICE == 'cpu':
    print(" WARNING: Training on CPU will be very slow!")

# Create directories
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results_v2', exist_ok=True)

# Load data
train_loader, val_loader, test_loader = create_dual_camera_dataloaders(
    DATASET_PATH, 
    batch_size=BATCH_SIZE,
    num_workers=0
)


CONSERVATIVE NOISE AUGMENTATION FOR UAV LOCALIZATION
Complete Research Pipeline V2.0

Device: cuda

STEP 1: LOADING DUAL-CAMERA DATASET

Loading cameras:
  Loading cam0 from C:\Users\Asus\Downloads\MH_01_easy\mav0...
    ✓ Loaded 3682 samples from cam0
  Loading cam1 from C:\Users\Asus\Downloads\MH_01_easy\mav0...
    ✓ Loaded 3682 samples from cam1

  ✓ Total samples: 7364
  ✓ Train: 5154 | Val: 1104 | Test: 1106
  ✓ Effective training size with augmentation: ~10308



In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  1. Baseline (no noise)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    'baseline': None,
    #'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    #'poisson': lambda img: noise_aug.poisson_noise(img, scale=200.0),
    #'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    #'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    #'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)



for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  2. Gaussian (σ=10)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    #'baseline': None,
    'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    #'poisson': lambda img: noise_aug.poisson_noise(img, scale=200.0),
    #'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    #'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    #'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)



for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  3. Poisson (scale=75)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    #'baseline': None,
    #'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    'poisson': lambda img: noise_aug.poisson_noise(img, scale=75.0),
    #'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    #'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    #'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)



for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  4. Motion Blur (kernel=5)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    #'baseline': None,
    #'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    #'poisson': lambda img: noise_aug.poisson_noise(img, scale=200.0),
    'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    #'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    #'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)



for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  5. Spatially Correlated (σ=10, corr=2)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    #'baseline': None,
    #'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    #'poisson': lambda img: noise_aug.poisson_noise(img, scale=200.0),
    #'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    #'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)



for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
# Define noise strategies
print("="*70)
print("STEP 2: DEFINING NOISE STRATEGIES")
print("="*70)
print("  6. Mixed (Poisson + Blur)")
print()

noise_aug = ConservativeNoiseAugmentation(seed=42)

noise_strategies = {
    # 'baseline': None,
    # 'gaussian': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    # 'poisson': lambda img: noise_aug.poisson_noise(img, scale=200.0),
    # 'motion_blur': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    # 'spatial': lambda img: noise_aug.spatially_correlated_noise(img, sigma=10.0, correlation_length=2),
    'mixed': lambda img: noise_aug.poisson_noise(noise_aug.motion_blur(img, kernel_size=5), scale=200.0),
}

# Train all models
print("="*70)
print("STEP 3: TRAINING ALL MODELS")
print("="*70)


for strategy_name, noise_fn in noise_strategies.items():
    model = StablePoseEstimator(pretrained=True)
    
    trained_model, history = train_model(
        model, train_loader, val_loader,
        noise_fn=noise_fn,
        model_name=strategy_name,
        num_epochs=NUM_EPOCHS,
        device=DEVICE
    )
    
    trained_models[strategy_name] = trained_model
    histories[strategy_name] = history

In [ ]:
import torch
import numpy as np
import pandas as pd
baseline_model=StablePoseEstimator().to(device=DEVICE)

baseline_model.load_state_dict(torch.load("History/baseline_best.pth"))
baseline_history=pd.read_csv(r"History/baseline_history")

In [ ]:
guassian_model=StablePoseEstimator().to(device=DEVICE)
guassian_model.load_state_dict(torch.load("History/gaussian_best.pth"))
guassian_history=pd.read_csv(r"History/guassan_history")

poisson_model=StablePoseEstimator().to(device=DEVICE)
poisson_model.load_state_dict(torch.load("History/poisson_best.pth"))
poisson_history=pd.read_csv(r"History/poisson_history")

motion_model=StablePoseEstimator().to(device=DEVICE)
motion_model.load_state_dict(torch.load("History/motion_blur_best.pth"))
motion_history=pd.read_csv(r"History/motion_history")

spatial_model=StablePoseEstimator().to(device=DEVICE)
spatial_model.load_state_dict(torch.load("History/spatial_best.pth"))
spatial_history=pd.read_csv(r"History/spatial_history")

mixed_model=StablePoseEstimator().to(device=DEVICE)
mixed_model.load_state_dict(torch.load("History/mixed_best.pth"))
mixed_history=pd.read_csv(r"History/mixed_history")


In [27]:
DEVICE

'cuda'

In [28]:
histories = {
    'baseline': baseline_history,
    'gaussian': guassian_history,
    'poisson': poisson_history,
    'motion_blur': motion_history,
    'spatial': spatial_history,
    'mixed': mixed_history,
}


trained_models = {
    'baseline': baseline_model,
    'gaussian': guassian_model,
    'poisson': poisson_model,
    'motion_blur': motion_model,
    'spatial': spatial_model,
    'mixed': mixed_model,
}

In [ ]:
def save_results(results, histories, save_dir):
    """
    Save all results: tables, plots, JSON.
    """

    # 1. Save results table
    with open(f'{save_dir}/results_table.txt', 'w') as f:
        f.write("="*120 + "\n")
        f.write("CONSERVATIVE NOISE AUGMENTATION - EVALUATION RESULTS\n")
        f.write("="*120 + "\n\n")
        
        f.write(f"{'Model':<20s} | {'Clean':<8s} | {'G-Low':<8s} | {'G-Med':<8s} | {'M-Blur':<8s} | {'Corr':<8s} | {'Mixed':<8s} | {'G-Ext':<8s} | {'P-Ext':<8s}\n")
        f.write("-"*120 + "\n")
        
        conditions_order = ['clean', 'gaussian_low', 'gaussian_medium', 'motion_blur_5', 
                           'correlated_noise', 'realistic_mixed', 'gaussian_extreme', 'poisson_extreme']
        
        for model_name, conditions in results.items():
            f.write(f"{model_name:<20s}")
            for cond in conditions_order:
                error = conditions[cond]['translation_error']
                f.write(f" | {error:>6.4f}m")
            f.write("\n")
        
        f.write("\nLower is better. Values are translation errors in meters.\n")
        f.write("G-Low=Gaussian Low, G-Med=Gaussian Medium, M-Blur=Motion Blur, Corr=Correlated,\n")
        f.write("Mixed=Realistic Mixed, G-Ext=Gaussian Extreme, P-Ext=Poisson Extreme\n")

    print("  ✓ Saved results_table.txt")

    # 2. Save JSON
    json_results = {}
    for model_name, conditions in results.items():
        json_results[model_name] = {}
        for cond_name, metrics in conditions.items():
            json_results[model_name][cond_name] = {
                'translation_error': float(metrics['translation_error']),
                'rotation_error': float(metrics['rotation_error'])
            }

    with open(f'{save_dir}/results.json', 'w') as f:
        json.dump(json_results, f, indent=2)

    print("  ✓ Saved results.json")

    # 3. Training curves
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, (name, history) in enumerate(histories.items()):
        ax = axes[idx]
        epochs = range(1, len(history['train_loss']) + 1)
        ax.plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
        ax.plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.set_title(name.replace('_', ' ').title(), fontweight='bold', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        best_epoch = np.argmin(history['val_loss'])
        ax.axvline(x=best_epoch+1, color='green', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    print("  ✓ Saved training_curves.png")

    # 4. Robustness comparison - grouped by noise type
    fig, ax = plt.subplots(figsize=(14, 6))

    conditions = ['clean', 'gaussian_low', 'gaussian_medium', 'motion_blur_5', 
                  'correlated_noise', 'realistic_mixed', 'gaussian_extreme', 'poisson_extreme']
    condition_labels = ['Clean', 'Gauss\nLow', 'Gauss\nMed', 'Motion\nBlur', 
                       'Corr\nNoise', 'Mixed\nReal', 'Gauss\nExtreme', 'Poisson\nExtreme']

    x = np.arange(len(conditions))
    width = 0.13

    colors = ['gray', 'blue', 'orange', 'green', 'red', 'purple']

    for idx, (model_name, color) in enumerate(zip(results.keys(), colors)):
        errors = [results[model_name][cond]['translation_error'] for cond in conditions]
        offset = (idx - 2.5) * width
        ax.bar(x + offset, errors, width, label=model_name, color=color, alpha=0.8)

    ax.set_xlabel('Test Condition', fontsize=12)
    ax.set_ylabel('Translation Error (meters)', fontsize=12)
    ax.set_title('Model Robustness Across Noise Conditions', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(condition_labels, fontsize=9)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    # Add vertical lines to separate noise categories
    ax.axvline(x=0.5, color='black', linestyle=':', alpha=0.3, linewidth=1)
    ax.axvline(x=2.5, color='black', linestyle=':', alpha=0.3, linewidth=1)
    ax.axvline(x=4.5, color='black', linestyle=':', alpha=0.3, linewidth=1)
    ax.axvline(x=5.5, color='black', linestyle=':', alpha=0.3, linewidth=1)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/robustness_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    print("  ✓ Saved robustness_comparison.png")

    # 5. Additional plot: Standard vs Extreme conditions
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Standard conditions (training-like)
    standard_conditions = ['clean', 'gaussian_low', 'gaussian_medium', 'motion_blur_5', 
                          'correlated_noise', 'realistic_mixed']
    standard_labels = ['Clean', 'G-Low', 'G-Med', 'M-Blur', 'Corr', 'Mixed']
    
    x_std = np.arange(len(standard_conditions))
    for idx, (model_name, color) in enumerate(zip(results.keys(), colors)):
        errors = [results[model_name][cond]['translation_error'] for cond in standard_conditions]
        offset = (idx - 2.5) * width
        ax1.bar(x_std + offset, errors, width, label=model_name, color=color, alpha=0.8)
    
    ax1.set_xlabel('Test Condition', fontsize=11)
    ax1.set_ylabel('Translation Error (meters)', fontsize=11)
    ax1.set_title('Standard/In-Distribution Tests', fontsize=12, fontweight='bold')
    ax1.set_xticks(x_std)
    ax1.set_xticklabels(standard_labels, fontsize=9)
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Extreme/OOD conditions
    extreme_conditions = ['gaussian_extreme', 'poisson_extreme']
    extreme_labels = ['Gaussian\nExtreme', 'Poisson\nExtreme']
    
    x_ext = np.arange(len(extreme_conditions))
    for idx, (model_name, color) in enumerate(zip(results.keys(), colors)):
        errors = [results[model_name][cond]['translation_error'] for cond in extreme_conditions]
        offset = (idx - 2.5) * width
        ax2.bar(x_ext + offset, errors, width, label=model_name, color=color, alpha=0.8)
    
    ax2.set_xlabel('Test Condition', fontsize=11)
    ax2.set_ylabel('Translation Error (meters)', fontsize=11)
    ax2.set_title('Extreme/Out-of-Distribution Tests', fontsize=12, fontweight='bold')
    ax2.set_xticks(x_ext)
    ax2.set_xticklabels(extreme_labels, fontsize=9)
    ax2.legend(loc='upper left', fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{save_dir}/standard_vs_extreme.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print("  ✓ Saved standard_vs_extreme.png")

    # 6. Summary statistics
    with open(f'{save_dir}/summary_statistics.txt', 'w') as f:
        f.write("="*70 + "\n")
        f.write("SUMMARY STATISTICS\n")
        f.write("="*70 + "\n\n")
        
        # Define condition groups
        standard_conditions = ['gaussian_low', 'gaussian_medium', 'motion_blur_5', 
                              'correlated_noise', 'realistic_mixed']
        extreme_conditions = ['gaussian_extreme', 'poisson_extreme']
        all_noisy = standard_conditions + extreme_conditions
        
        # Average errors on different condition groups
        f.write("Average Performance:\n")
        avg_errors_standard = {}
        avg_errors_extreme = {}
        avg_errors_all = {}
        
        for model_name, conditions in results.items():
            avg_std = np.mean([conditions[cond]['translation_error'] 
                              for cond in standard_conditions])
            avg_ext = np.mean([conditions[cond]['translation_error'] 
                              for cond in extreme_conditions])
            avg_all = np.mean([conditions[cond]['translation_error'] 
                              for cond in all_noisy])
            
            avg_errors_standard[model_name] = avg_std
            avg_errors_extreme[model_name] = avg_ext
            avg_errors_all[model_name] = avg_all
            
            f.write(f"  {model_name:20s}:\n")
            f.write(f"    Standard conditions: {avg_std:.4f}m\n")
            f.write(f"    Extreme conditions:  {avg_ext:.4f}m\n")
            f.write(f"    All noisy:          {avg_all:.4f}m\n\n")
        
        # Best models
        best_standard = min(avg_errors_standard, key=avg_errors_standard.get)
        best_extreme = min(avg_errors_extreme, key=avg_errors_extreme.get)
        best_overall = min(avg_errors_all, key=avg_errors_all.get)
        
        f.write("Best Models:\n")
        f.write(f"  Standard conditions: {best_standard} ({avg_errors_standard[best_standard]:.4f}m)\n")
        f.write(f"  Extreme conditions:  {best_extreme} ({avg_errors_extreme[best_extreme]:.4f}m)\n")
        f.write(f"  Overall (all noisy): {best_overall} ({avg_errors_all[best_overall]:.4f}m)\n\n")
        
        # Best on each condition
        f.write("Best Model Per Condition:\n")
        all_conditions = ['clean', 'gaussian_low', 'gaussian_medium', 'motion_blur_5',
                         'correlated_noise', 'realistic_mixed', 'gaussian_extreme', 'poisson_extreme']
        condition_display = {
            'clean': 'Clean',
            'gaussian_low': 'Gaussian Low',
            'gaussian_medium': 'Gaussian Medium',
            'motion_blur_5': 'Motion Blur',
            'correlated_noise': 'Correlated Noise',
            'realistic_mixed': 'Realistic Mixed',
            'gaussian_extreme': 'Gaussian Extreme',
            'poisson_extreme': 'Poisson Extreme'
        }
        
        for cond in all_conditions:
            errors = {name: results[name][cond]['translation_error'] 
                     for name in results.keys()}
            best = min(errors, key=errors.get)
            f.write(f"  {condition_display[cond]:20s}: {best:20s} ({errors[best]:.4f}m)\n")
        
        f.write("\n")
        
        # Improvement over baseline
        f.write("Improvement Over Baseline:\n")
        baseline_errors_std = {cond: results['baseline'][cond]['translation_error'] 
                               for cond in standard_conditions}
        baseline_errors_ext = {cond: results['baseline'][cond]['translation_error'] 
                               for cond in extreme_conditions}
        
        for model_name in results.keys():
            if model_name == 'baseline':
                continue
            
            improvements_std = []
            improvements_ext = []
            
            for cond in standard_conditions:
                baseline_err = baseline_errors_std[cond]
                model_err = results[model_name][cond]['translation_error']
                improvement = ((baseline_err - model_err) / baseline_err) * 100
                improvements_std.append(improvement)
            
            for cond in extreme_conditions:
                baseline_err = baseline_errors_ext[cond]
                model_err = results[model_name][cond]['translation_error']
                improvement = ((baseline_err - model_err) / baseline_err) * 100
                improvements_ext.append(improvement)
            
            avg_improvement_std = np.mean(improvements_std)
            avg_improvement_ext = np.mean(improvements_ext)
            avg_improvement_all = np.mean(improvements_std + improvements_ext)
            
            f.write(f"  {model_name:20s}:\n")
            f.write(f"    Standard: {avg_improvement_std:+6.1f}%\n")
            f.write(f"    Extreme:  {avg_improvement_ext:+6.1f}%\n")
            f.write(f"    Overall:  {avg_improvement_all:+6.1f}%\n\n")
        
        # Critical test performance (realistic_mixed)
        f.write("Critical Test (Realistic Mixed Noise):\n")
        mixed_errors = {name: results[name]['realistic_mixed']['translation_error'] 
                       for name in results.keys()}
        best_mixed = min(mixed_errors, key=mixed_errors.get)
        f.write(f"  Best: {best_mixed} ({mixed_errors[best_mixed]:.4f}m)\n")
        for model_name, error in sorted(mixed_errors.items(), key=lambda x: x[1]):
            improvement = ((mixed_errors['baseline'] - error) / mixed_errors['baseline']) * 100
            f.write(f"  {model_name:20s}: {error:.4f}m ({improvement:+5.1f}%)\n")
        f.write("\n")
        
        # Training stability
        f.write("Training Stability (Best Epoch):\n")
        for model_name, history in histories.items():
            best_epoch = np.argmin(history['val_loss']) + 1
            total_epochs = len(history['val_loss'])
            f.write(f"  {model_name:20s}: Epoch {best_epoch:2d}/{total_epochs}\n")

    print("  ✓ Saved summary_statistics.txt")
    print()

In [31]:
# Evaluate all models
print("="*70)
print("STEP 4: EVALUATING ALL MODELS")
print("="*70 + "\n")

# Define test conditions
test_conditions = {
    'clean': None,
    # Standard Additive Noise Tests
    'gaussian_low': lambda img: noise_aug.gaussian_noise(img, sigma=5.0),
    'gaussian_medium': lambda img: noise_aug.gaussian_noise(img, sigma=10.0),
    # Structural Degradation Tests (Based on your training)
    'motion_blur_5': lambda img: noise_aug.motion_blur(img, kernel_size=5),
    'correlated_noise': lambda img: noise_aug.spatially_correlated_noise(img, sigma=5.0, correlation_length=2),
    # Realistic Mixed Noise Test (Critical)
    'realistic_mixed': lambda img: noise_aug.gaussian_noise(noise_aug.poisson_noise(img, scale=75.0), sigma=5.0),
    # Out-of-Distribution/Extreme Tests (To check for overfitting)
    'gaussian_extreme': lambda img: noise_aug.gaussian_noise(img, sigma=20.0), 
    'poisson_extreme': lambda img: noise_aug.poisson_noise(img, scale=250.0),
}



for model_name, model in trained_models.items():
    print(f"Evaluating {model_name}...")
    results[model_name] = {}
    
    for condition_name, test_noise_fn in test_conditions.items():
        metrics = evaluate_model(model, test_loader, test_noise_fn, DEVICE, condition_name)
        results[model_name][condition_name] = metrics
        print(f"  {condition_name:15s}: {metrics['translation_error']:.4f}m")

STEP 4: EVALUATING ALL MODELS

Evaluating baseline...
  clean          : 1.0831m
  gaussian_low   : 2.7510m
  gaussian_medium: 2.7884m
  motion_blur_5  : 2.1447m
  correlated_noise: 1.1361m
  realistic_mixed: 5.5384m
  gaussian_extreme: 2.9686m
  poisson_extreme: 6.7802m
Evaluating gaussian...
  clean          : 1.3345m
  gaussian_low   : 1.3232m
  gaussian_medium: 1.2698m
  motion_blur_5  : 2.0233m
  correlated_noise: 1.2842m
  realistic_mixed: 4.7724m
  gaussian_extreme: 1.4154m
  poisson_extreme: 4.3314m
Evaluating poisson...
  clean          : 4.0017m
  gaussian_low   : 4.1593m
  gaussian_medium: 4.1540m
  motion_blur_5  : 4.1218m
  correlated_noise: 4.0007m
  realistic_mixed: 1.7148m
  gaussian_extreme: 4.1282m
  poisson_extreme: 2.5258m
Evaluating motion_blur...
  clean          : 1.9098m
  gaussian_low   : 2.7969m
  gaussian_medium: 2.8307m
  motion_blur_5  : 1.2089m
  correlated_noise: 1.9586m
  realistic_mixed: 5.2450m
  gaussian_extreme: 3.0542m
  poisson_extreme: 6.2701m
Eva

In [34]:
# Save results
print("\n" + "="*70)
print("STEP 5: SAVING RESULTS")
print("="*70 + "\n")
save_results(results, histories, 'results_v2')


STEP 5: SAVING RESULTS

  ✓ Saved results_table.txt
  ✓ Saved results.json
  ✓ Saved training_curves.png
  ✓ Saved robustness_comparison.png
  ✓ Saved standard_vs_extreme.png
  ✓ Saved summary_statistics.txt

